# Protein Subcellular Localization — ESM-2 Scaling
**Runtime:** T4 GPU (Runtime → Change runtime type → T4 GPU)

All heavy steps checkpoint to `/content/checkpoints/` so disconnections don't lose progress.

In [ ]:
# ── 0. setup (quick — already installed pkgs are skipped) ────────
!pip install -q transformers xgboost umap-learn 2>/dev/null
import torch, os, gc, time, json, warnings
import numpy as np, pandas as pd
from pathlib import Path
from collections import Counter
warnings.filterwarnings('ignore')

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {dev}', f'| gpu: {torch.cuda.get_device_name(0)}' if dev=='cuda' else '| NO GPU')

# dirs
root = Path('/content/protein-loc-scaling')
ckpt = Path('/content/checkpoints'); ckpt.mkdir(exist_ok=True)
data_dir = root / 'data'; data_dir.mkdir(parents=True, exist_ok=True)
emb_dir = ckpt / 'embeddings'; emb_dir.mkdir(exist_ok=True)
res_dir = ckpt / 'results'; res_dir.mkdir(exist_ok=True)

# clone repo if needed
if not root.exists():
    !git clone https://github.com/r-siddiqi/protein-loc-scaling.git {root}
print('setup done')

In [ ]:
# ── 1. download deeploc 2.0 (skips if already on disk) ──────────
# dtu test url is often down — we use the train csv's Partition column
# to create our own 80/20 stratified split instead. this is actually
# more reproducible since the dtu test set csv has different columns.
import urllib.request

train_url = 'https://services.healthtech.dtu.dk/services/DeepLoc-2.0/data/Swissprot_Train_Validation_dataset.csv'
p = data_dir / 'deeploc_train.csv'

if p.exists():
    print('train csv: cached')
else:
    try:
        urllib.request.urlretrieve(train_url, p)
        print(f'train csv: downloaded from dtu ({p.stat().st_size/1e6:.1f} MB)')
    except Exception as e:
        print(f'dtu failed: {e}')
        # fallback: use huggingface datasets library
        print('trying huggingface datasets...')
        !pip install -q datasets
        from datasets import load_dataset
        ds = load_dataset('bloyal/deeploc', split='train')
        ds.to_pandas().to_csv(p, index=False)
        print(f'train csv: downloaded from hf ({p.stat().st_size/1e6:.1f} MB)')

# peek at columns so we know what we're working with
df_peek = pd.read_csv(p, nrows=3)
print(f'columns: {df_peek.columns.tolist()}')

In [ ]:
# ── 2. parse & filter sequences ─────────────────────────────────
# deeploc 2.0 csv uses ONE-HOT columns for each compartment, not a
# single "location" column. a protein can have multiple labels (multi-label).
# we keep only single-label proteins for clean 10-class classification.
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

labels = ['Nucleus','Cytoplasm','Extracellular','Mitochondrion',
          'Cell membrane','Endoplasmic reticulum','Plastid',
          'Golgi apparatus','Lysosome/Vacuole','Peroxisome']
max_len, min_len = 1022, 30
ok_aa = set('ACDEFGHIKLMNPQRSTVWY')
seed = 42

df = pd.read_csv(data_dir / 'deeploc_train.csv')
print(f'raw csv: {len(df)} rows, columns: {df.columns.tolist()[:8]}...')

# find the sequence column (case-insensitive)
df.columns = [c.strip() for c in df.columns]
seq_col = next((c for c in df.columns if c.lower() in ('sequence','seq')), None)
if not seq_col:
    raise ValueError(f'no sequence column found in: {df.columns.tolist()}')

# detect format: one-hot columns vs single label column
loc_cols = [c for c in labels if c in df.columns]
single_loc_col = next((c for c in df.columns if c.lower() in ('location','localization','label')), None)

if loc_cols:
    # ── one-hot format (standard deeploc 2.0) ──
    print(f'detected one-hot format with {len(loc_cols)} location columns')
    # for each row, find which locations are 1
    df['_loc_list'] = df[loc_cols].apply(lambda r: [c for c in loc_cols if r[c]==1], axis=1)
    df['_n_locs'] = df['_loc_list'].apply(len)
    # keep only single-label proteins
    n_multi = (df['_n_locs'] > 1).sum()
    n_zero = (df['_n_locs'] == 0).sum()
    print(f'  single-label: {(df["_n_locs"]==1).sum()} | multi-label: {n_multi} | no-label: {n_zero}')
    df = df[df['_n_locs'] == 1].copy()
    df['location'] = df['_loc_list'].apply(lambda x: x[0])
elif single_loc_col:
    # ── single location column ──
    print(f'detected single-label format (column: {single_loc_col})')
    df['location'] = df[single_loc_col].astype(str).str.strip()
else:
    raise ValueError(f'could not find location columns. available: {df.columns.tolist()}')

# filter by sequence quality and length
accs, locs, seqs = [], [], []
n_short, n_long, n_bad_aa, n_bad_label = 0, 0, 0, 0

id_col = next((c for c in df.columns if c.lower() in ('acc','id','accession','protein_id','entry')),
              df.columns[0])

for _, r in df.iterrows():
    sq = str(r[seq_col]).upper().strip()
    lo = str(r['location']).strip()
    if len(sq) < min_len: n_short += 1; continue
    if len(sq) > max_len: n_long += 1; continue
    if not all(c in ok_aa for c in sq): n_bad_aa += 1; continue
    if lo not in labels: n_bad_label += 1; continue
    accs.append(str(r[id_col]).strip())
    locs.append(lo)
    seqs.append(sq)

print(f'\nkept {len(seqs)}/{len(df)} after filtering')
print(f'  short={n_short}, long={n_long}, bad_aa={n_bad_aa}, bad_label={n_bad_label}')

# ── stratified train/test split (80/20) ──
# since dtu test csv is unreliable, we create our own split
idx_tr, idx_te = train_test_split(
    range(len(seqs)), test_size=0.2, random_state=seed,
    stratify=locs)

tr_acc = [accs[i] for i in idx_tr]; tr_loc = [locs[i] for i in idx_tr]; tr_seq = [seqs[i] for i in idx_tr]
te_acc = [accs[i] for i in idx_te]; te_loc = [locs[i] for i in idx_te]; te_seq = [seqs[i] for i in idx_te]

le = LabelEncoder(); le.fit(labels)
y_tr = le.transform(tr_loc); y_te = le.transform(te_loc)

print(f'\ntrain: {len(tr_seq)} | test: {len(te_seq)}')
print('\nclass distribution (train):')
for c in labels: print(f'  {c:30s}: {tr_loc.count(c):5d} ({100*tr_loc.count(c)/len(tr_loc):.1f}%)')

## 3. Extract ESM-2 Embeddings (checkpointed per model+split)

In [ ]:
# ── embedding extraction with per-model checkpointing ───────────
from transformers import AutoTokenizer, AutoModel

models = {
    'esm2_8m':   ('facebook/esm2_t6_8M_UR50D',    320),
    'esm2_35m':  ('facebook/esm2_t12_35M_UR50D',   480),
    'esm2_150m': ('facebook/esm2_t30_150M_UR50D',   640),
    'esm2_650m': ('facebook/esm2_t33_650M_UR50D',  1280),
}

def extract(seqs, model_name):
    """mean-pooled last hidden layer, batch_size=1, float32."""
    hf_id, dim = models[model_name]
    tok = AutoTokenizer.from_pretrained(hf_id)
    mdl = AutoModel.from_pretrained(hf_id, torch_dtype=torch.float32).to(dev).eval()
    n = len(seqs); emb = np.zeros((n, dim), dtype=np.float32); t0 = time.time()
    with torch.no_grad():
        for i, sq in enumerate(seqs):
            inp = tok(sq[:max_len], return_tensors='pt', padding=False,
                      truncation=True, max_length=max_len+2).to(dev)
            h = mdl(**inp).last_hidden_state
            emb[i] = h[0, 1:-1, :].mean(0).cpu().numpy()
            if (i+1)%200==0 or i==n-1:
                r = (i+1)/(time.time()-t0)
                print(f'  [{i+1}/{n}] {r:.1f} seq/s, eta {(n-i-1)/r:.0f}s')
            if (i+1)%500==0 and dev=='cuda': torch.cuda.empty_cache()
    del mdl, tok; gc.collect()
    if dev=='cuda': torch.cuda.empty_cache()
    return emb

# extract all, with checkpoint skipping
all_emb = {}  # {model: {'train': arr, 'test': arr}}

for mn in models:
    print(f'\n{"="*50}\n{mn}\n{"="*50}')
    all_emb[mn] = {}
    d = emb_dir / mn; d.mkdir(exist_ok=True)
    for split, seqs in [('train', tr_seq), ('test', te_seq)]:
        p = d / f'{split}.npy'
        if p.exists():
            print(f'  {split}: loading checkpoint')
            all_emb[mn][split] = np.load(p)
        else:
            print(f'  {split}: extracting...')
            e = extract(seqs, mn)
            np.save(p, e)  # checkpoint!
            print(f'  {split}: saved checkpoint → {p}')
            all_emb[mn][split] = e
        print(f'  {split}: {all_emb[mn][split].shape}')

print('\nall embeddings ready!')

## 4. Train Classifiers (checkpointed per model × classifier)

In [ ]:
# ── classifier defs ─────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, matthews_corrcoef,
                             confusion_matrix, roc_auc_score, roc_curve, auc)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

def get_clfs():
    """return dict of (pipeline, param_grid) for each classifier."""
    return {
        'logistic_regression': (
            Pipeline([('sc',StandardScaler()),('clf',LogisticRegression(
                max_iter=2000,class_weight='balanced',solver='lbfgs',
                multi_class='multinomial',random_state=seed))]),
            {'clf__C':[0.01,0.1,1.0,10.0]}),
        'random_forest': (
            Pipeline([('sc',StandardScaler()),('clf',RandomForestClassifier(
                class_weight='balanced',n_jobs=-1,random_state=seed))]),
            {'clf__n_estimators':[100,300],'clf__max_depth':[None,20]}),
        'svm': (
            Pipeline([('sc',StandardScaler()),('clf',SVC(
                kernel='rbf',class_weight='balanced',probability=True,random_state=seed))]),
            {'clf__C':[0.1,1.0,10.0]}),
        'knn': (
            Pipeline([('sc',StandardScaler()),('clf',KNeighborsClassifier(
                weights='distance',metric='cosine'))]),
            {'clf__n_neighbors':[5,11,21]}),
        'xgboost': (
            Pipeline([('sc',StandardScaler()),('clf',XGBClassifier(
                subsample=0.8,colsample_bytree=0.8,eval_metric='mlogloss',
                random_state=seed,verbosity=0))]),
            {'clf__n_estimators':[100,300],'clf__max_depth':[4,6],
             'clf__learning_rate':[0.01,0.1]}),
        'mlp': (
            Pipeline([('sc',StandardScaler()),('clf',MLPClassifier(
                activation='relu',early_stopping=True,validation_fraction=0.1,
                max_iter=500,random_state=seed))]),
            {'clf__hidden_layer_sizes':[(256,),(256,128)],
             'clf__learning_rate_init':[0.001,0.0001]}),
    }

print('classifiers defined')

In [ ]:
# ── train with per-model×classifier checkpoints ──────────────────
# each (model, classifier) pair is saved independently so a
# disconnect only loses the single classifier being trained.

all_res = {}  # {model: {clf: metrics}}

for mn in models:
    print(f'\n{"="*60}\n{mn}\n{"="*60}')
    x_tr = all_emb[mn]['train']; x_te = all_emb[mn]['test']
    print(f'train: {x_tr.shape}, test: {x_te.shape}')
    
    model_res = {}
    model_ckpt_dir = res_dir / mn; model_ckpt_dir.mkdir(exist_ok=True)
    
    for cn, (pipe, grid) in get_clfs().items():
        ckpt_file = model_ckpt_dir / f'{cn}.json'
        
        # ── checkpoint: skip if already trained ──
        if ckpt_file.exists():
            print(f'  {cn}: loading checkpoint')
            with open(ckpt_file) as f:
                model_res[cn] = json.load(f)
            print(f'    macro_f1={model_res[cn]["macro_f1"]:.4f}')
            continue
        
        print(f'  {cn}: training...')
        t0 = time.time()
        
        gs = GridSearchCV(pipe, grid, cv=cv, scoring='f1_macro', n_jobs=-1, refit=True, verbose=0)
        gs.fit(x_tr, y_tr)
        
        yp = gs.predict(x_te)
        ypr = gs.predict_proba(x_te) if hasattr(gs.best_estimator_,'predict_proba') else None
        
        # metrics
        acc = accuracy_score(y_te, yp)
        mf1 = f1_score(y_te, yp, average='macro', zero_division=0)
        wf1 = f1_score(y_te, yp, average='weighted', zero_division=0)
        pcm = {}
        for i,nm in enumerate(labels):
            bt=(y_te==i).astype(int); bp=(yp==i).astype(int)
            pcm[nm] = matthews_corrcoef(bt,bp) if bt.sum()>0 else 0.0
        amcc = np.mean(list(pcm.values()))
        cm = confusion_matrix(y_te, yp, labels=list(range(len(labels))))
        rauc = None
        if ypr is not None:
            try:
                yb = label_binarize(y_te, classes=list(range(len(labels))))
                rauc = float(roc_auc_score(yb, ypr, average='macro', multi_class='ovr'))
            except: pass
        
        elapsed = time.time()-t0
        
        r = {
            'accuracy':acc,'macro_f1':mf1,'weighted_f1':wf1,'avg_mcc':amcc,
            'per_class_mcc':pcm,'confusion_matrix':cm.tolist(),'roc_auc':rauc,
            'best_params':{k.replace('clf__',''):v for k,v in gs.best_params_.items()},
            'best_cv_score':float(gs.best_score_),
            'y_pred':yp.tolist(),'y_prob':ypr.tolist() if ypr is not None else None,
        }
        model_res[cn] = r
        
        # ── checkpoint: save immediately after each classifier ──
        with open(ckpt_file, 'w') as f:
            json.dump(r, f, indent=2)
        print(f'    macro_f1={mf1:.4f} | acc={acc:.4f} | mcc={amcc:.4f} | '
              f'cv={gs.best_score_:.4f} | {elapsed:.0f}s | ✓ saved')
    
    all_res[mn] = model_res

print('\n\nall training complete!')

In [ ]:
# ── summary table ───────────────────────────────────────────────
print(f'{"model":<15} {"classifier":<22} {"macro_f1":>9} {"acc":>9} {"mcc":>9} {"auc":>9}')
print('─'*70)
for mn in models:
    if mn not in all_res: continue
    for cn,m in sorted(all_res[mn].items(), key=lambda x:-x[1]['macro_f1']):
        a = f"{m['roc_auc']:.4f}" if m['roc_auc'] else 'n/a'
        print(f"{mn:<15} {cn:<22} {m['macro_f1']:>9.4f} {m['accuracy']:>9.4f} {m['avg_mcc']:>9.4f} {a:>9}")
    print()

## 5. Visualizations

In [ ]:
# ── scaling curves ──────────────────────────────────────────────
import matplotlib.pyplot as plt, matplotlib, seaborn as sns
matplotlib.rcParams.update({'font.size':11,'figure.dpi':150})

msz = {'esm2_8m':8,'esm2_35m':35,'esm2_150m':150,'esm2_650m':650}
cc = {'logistic_regression':'#1f77b4','random_forest':'#2ca02c',
      'svm':'#d62728','knn':'#9467bd','xgboost':'#ff7f0e','mlp':'#8c564b'}

fig,axes = plt.subplots(1,3,figsize=(18,5))
for ax,met,ttl in zip(axes,['macro_f1','accuracy','avg_mcc'],['Macro F1','Accuracy','Avg MCC']):
    clfs = set(); [clfs.update(v.keys()) for v in all_res.values()]
    for cn in sorted(clfs):
        xs,ys = [],[]
        for mn in ['esm2_8m','esm2_35m','esm2_150m','esm2_650m']:
            if mn in all_res and cn in all_res[mn]:
                xs.append(msz[mn]); ys.append(all_res[mn][cn][met])
        if xs: ax.plot(xs,ys,'o-',label=cn.replace('_',' ').title(),
                       color=cc.get(cn,'#333'),lw=2,ms=6)
    ax.set_xscale('log'); ax.set_xticks([8,35,150,650])
    ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
    ax.set_xlabel('Model Size (M params)'); ax.set_ylabel(ttl)
    ax.set_title(f'{ttl} vs Model Size'); ax.legend(fontsize=7); ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(ckpt/'scaling_curves.png',dpi=300,bbox_inches='tight'); plt.show()

In [ ]:
# ── confusion matrices (best clf per model) ─────────────────────
nm = len(all_res)
fig,axes = plt.subplots(1,nm,figsize=(7*nm,6))
if nm==1: axes=[axes]
for ax,(mn,cr) in zip(axes,all_res.items()):
    bc = max(cr,key=lambda k:cr[k]['macro_f1'])
    cm = np.array(cr[bc]['confusion_matrix'])
    cm_n = cm.astype(float)/cm.sum(axis=1,keepdims=True); cm_n=np.nan_to_num(cm_n)
    sns.heatmap(cm_n,annot=True,fmt='.2f',cmap='Blues',
                xticklabels=labels,yticklabels=labels,ax=ax,vmin=0,vmax=1)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'{mn} + {bc.replace("_"," ").title()}')
    ax.tick_params(axis='x',rotation=45); ax.tick_params(axis='y',rotation=0)
plt.tight_layout()
plt.savefig(ckpt/'confusion_matrices.png',dpi=300,bbox_inches='tight'); plt.show()

In [ ]:
# ── pca of embedding space ──────────────────────────────────────
from sklearn.decomposition import PCA
nm = len(all_res)
fig,axes = plt.subplots(1,nm,figsize=(7*nm,6))
if nm==1: axes=[axes]
for ax,mn in zip(axes,all_res.keys()):
    x = all_emb[mn]['test']
    pca = PCA(n_components=2,random_state=seed); co = pca.fit_transform(x)
    for i,nm_ in enumerate(labels):
        m = y_te==i
        if m.sum()>0: ax.scatter(co[m,0],co[m,1],s=5,alpha=0.4,label=nm_)
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title(f'{mn} PCA'); ax.legend(fontsize=6,markerscale=3); ax.grid(True,alpha=0.2)
plt.tight_layout()
plt.savefig(ckpt/'pca_embeddings.png',dpi=300,bbox_inches='tight'); plt.show()

In [ ]:
# ── per-class mcc bar chart (largest model) ─────────────────────
bm = list(all_res.keys())[-1]  # largest available model
cr = all_res[bm]; cns = sorted(cr.keys())
nc = len(labels); ncl = len(cns)
fig,ax = plt.subplots(figsize=(14,5))
x = np.arange(nc); w = 0.8/ncl
for i,cn in enumerate(cns):
    vals = [float(cr[cn]['per_class_mcc'].get(l,0)) for l in labels]
    ax.bar(x+(i-ncl/2+0.5)*w, vals, w, label=cn.replace('_',' ').title(),
           color=cc.get(cn,'#333'), alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labels,rotation=45,ha='right')
ax.set_ylabel('MCC'); ax.set_title(f'Per-Class MCC — {bm}')
ax.legend(fontsize=7); ax.grid(True,axis='y',alpha=0.3); ax.axhline(0,color='k',lw=0.5)
plt.tight_layout()
plt.savefig(ckpt/'per_class_mcc.png',dpi=300,bbox_inches='tight'); plt.show()

In [ ]:
# ── roc curves (best model + best classifier) ───────────────────
bm = list(all_res.keys())[-1]
bc = max(all_res[bm],key=lambda k:all_res[bm][k]['macro_f1'])
ypr = np.array(all_res[bm][bc]['y_prob'])
yb = label_binarize(y_te, classes=list(range(len(labels))))

fig,ax = plt.subplots(figsize=(8,7))
if ypr is not None:
    for i,nm in enumerate(labels):
        if yb[:,i].sum()==0: continue
        fpr,tpr,_ = roc_curve(yb[:,i],ypr[:,i]); a = auc(fpr,tpr)
        ax.plot(fpr,tpr,label=f'{nm} ({a:.3f})',lw=1.5)
ax.plot([0,1],[0,1],'k--',lw=0.8,alpha=0.5)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title(f'ROC — {bm} + {bc.replace("_"," ").title()}')
ax.legend(loc='lower right',fontsize=8); ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(ckpt/'roc_curves.png',dpi=300,bbox_inches='tight'); plt.show()

## 6. Statistical Tests

In [ ]:
# ── bootstrap 95% CI for macro f1 ───────────────────────────────
rng = np.random.RandomState(seed); nb = 1000
print(f'{"model":<15} {"classifier":<22} {"macro_f1":>9} {"95% CI":>20}')
print('─'*70)
for mn in all_res:
    for cn,m in sorted(all_res[mn].items(),key=lambda x:-x[1]['macro_f1']):
        yp = np.array(m['y_pred'])
        sc = [f1_score(y_te[ix:=rng.choice(len(y_te),len(y_te),replace=True)],
                       yp[ix],average='macro',zero_division=0) for _ in range(nb)]
        lo,hi = np.percentile(sc,[2.5,97.5])
        print(f"{mn:<15} {cn:<22} {m['macro_f1']:>9.4f} [{lo:.4f}, {hi:.4f}]")
    print()

In [ ]:
# ── wilcoxon signed-rank (per-class f1 between adjacent models) ──
from scipy import stats
mo = [m for m in ['esm2_8m','esm2_35m','esm2_150m','esm2_650m'] if m in all_res]
clfs = set(); [clfs.update(v.keys()) for v in all_res.values()]

print('Wilcoxon Signed-Rank Tests (per-class F1)\n'+'-'*60)
for cn in sorted(clfs):
    print(f'\n  {cn}:')
    for i in range(len(mo)-1):
        a,b = mo[i],mo[i+1]
        fa = f1_score(y_te,np.array(all_res[a][cn]['y_pred']),average=None,zero_division=0)
        fb = f1_score(y_te,np.array(all_res[b][cn]['y_pred']),average=None,zero_division=0)
        d = fb-fa
        if np.all(d==0): print(f'    {a} → {b}: identical'); continue
        try:
            _,p = stats.wilcoxon(fa,fb)
            print(f'    {a} → {b}: p={p:.4f}{" *" if p<0.05 else "  "} (Δ={np.mean(d):+.4f})')
        except Exception as e: print(f'    {a} → {b}: err ({e})')
print('\n* = p < 0.05')

## 7. Download Results

In [ ]:
# ── zip everything and download ─────────────────────────────────
!cd /content && zip -r protein_loc_results.zip checkpoints/
from google.colab import files
files.download('/content/protein_loc_results.zip')
print('download started!')